In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
df = pd.read_csv("/Volumes/squirrel-utopia 1/national_wf_disaster_hosp/local_data/exposed_population_counts_by_zcta.csv") 

In [ ]:
# Create lookup DataFrame for month index to date string
month_lookup = pd.DataFrame({
    "month": range(1, 229),  # 1 to 228
    "month_str": pd.date_range("2000-01-01", periods=228, freq="MS").strftime("%Y-%m")
})

# Left join to add month_str to your df
df = df.merge(month_lookup, on="month", how="left")

In [ ]:
print(df)

In [ ]:
gdf = gpd.read_file("/Volumes/squirrel-utopia 1/national_wf_disaster_hosp/local_data/zctas_2020.geojson")

In [ ]:
gdf.plot()

In [ ]:
print(gdf)

In [ ]:

df["ID_admin_unit"] = df["ID_admin_unit"].astype(str)
df["year"] = df["month_str"].str[:4].astype(int)
print(df)



In [ ]:
df.head()
df["ID_admin_unit"] = df["ID_admin_unit"].astype(str)
gdf["ID_admin_unit"] = gdf["ID_admin_unit"].astype(str)


In [ ]:
dfs_by_month = []
for m in df["month"].unique():
    dfs_by_month.append(df[df["month"] == m])

In [ ]:
z = dfs_by_month[0]
print(z.dtypes)
print(gdf.dtypes)


In [ ]:
merged_by_month = []
for i in range(1, len(dfs_by_month)):
    merged = gdf.merge(dfs_by_month[i], how="left")
    for col in ["exposed_main", "exposed_larger", "exposed_smaller"]:
        merged[col] = merged[col].fillna(0)
        # Fill month and month_str columns
    month_val = dfs_by_month[i]["month"].dropna().unique()[0]
    month_str_val = dfs_by_month[i]["month_str"].dropna().unique()[0]
    merged["month"] = month_val
    merged["month_str"] = month_str_val
    merged = merged[merged['ID_admin_unit'].str[:3].astype(int).between(900, 961)]
    merged_by_month.append(merged)

In [ ]:

oo = merged_by_month[220]
print(oo)

In [ ]:
with PdfPages(f"test_maps.pdf") as pdf:
    plotted_df = merged_by_month[0]
    t = plotted_df["month_str"].unique()[0]
    fig, ax = plt.subplots(figsize=(8, 8))
    plotted_df.plot(column='exposed_main', ax=ax, legend=True, cmap="viridis")
    ax.set_title(f"exposed_main at time {t}")
    ax.axis("off")
    pdf.savefig(fig)
    plt.close(fig)

In [ ]:
from tqdm import tqdm

# List of value columns
value_cols = ["exposed_main"]  # Add others if needed

for value in value_cols:
    with PdfPages(f"{value}_maps.pdf") as pdf:
        for merged in tqdm(merged_by_month, desc=f"Mapping {value}"):
            t = merged["month_str"].unique()[0]
            fig, ax = plt.subplots(figsize=(8, 8))
            merged.plot(column=value, ax=ax, legend=True, cmap="viridis")
            ax.set_title(f"{value} at time {t}")
            ax.axis("off")
            pdf.savefig(fig)
            plt.close(fig)

In [ ]:
# List your value columns
value_cols = ["exposed_main", "exposed_larger", "exposed_smaller"]  # adjust as needed

# Group by year and sum
yearly_sum = df.groupby(["year", "ID_admin_unit"])[value_cols].sum().reset_index()

print(yearly_sum)

In [ ]:
dfs_by_year = []
for m in df["year"].unique():
    dfs_by_year.append(df[df["year"] == m])

In [ ]:
dfs_by_year[0]

In [ ]:
merged_by_year = []
for i in range(0, len(dfs_by_year)):
    merged = gdf.merge(dfs_by_year[i], how="left")
    for col in ["exposed_main", "exposed_larger", "exposed_smaller"]:
        merged[col] = merged[col].fillna(0)
        # Fill month and month_str columns
    year_val = dfs_by_year[i]["year"].dropna().unique()[0]
    merged["year"] = year_val
    merged = merged[merged['ID_admin_unit'].str[:3].astype(int).between(900, 961)]
    merged_by_year.append(merged)

In [ ]:
merged_by_year[0]

In [ ]:
from tqdm import tqdm

# List of value columns
value_cols = ["exposed_main"]  # Add others if needed

for value in value_cols:
    with PdfPages(f"{value}_maps_yearly.pdf") as pdf:
        for merged in tqdm(merged_by_year, desc=f"Mapping {value}"):
            t = merged["year"].unique()[0]
            fig, ax = plt.subplots(figsize=(8, 8))
            merged.plot(column=value, ax=ax, legend=True, cmap="viridis")
            ax.set_title(f"{value} at time {t}")
            ax.axis("off")
            pdf.savefig(fig, dpi=50)
            plt.close(fig)